In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Le fichier config_projet.py doit se trouver dans E:\Projet_Alzheimer\configs
sys.path.append(r"E:\Projet_Alzheimer\configs")
from config_projet import C

print("Cohorte 359   :", C.COHORTE.name)
print("Audit visites :", C.AUDIT_VISITES.name)

participants_359 = pd.read_csv(C.PARTICIPANTS_359, dtype={"PTID": "string"})
ptids_359 = set(participants_359["PTID"].str.strip().str.upper())
print("Participants cohorte complete :", len(ptids_359), "(attendu : 359)")

manifeste_112 = pd.read_csv(C.MANIFESTE_112, dtype="string")
print("Participants groupe MMSE-1an  :",
      manifeste_112["PTID"].nunique(), "(attendu : 112)")

In [ ]:
# A.1 Historique diagnostique des 359 participants.
dx = pd.read_csv(C.table("All_Subjects_DXSUM_"), dtype="string", encoding="utf-8-sig")
dx["PTID"] = dx["PTID"].str.strip().str.upper()
dx["DIAGNOSIS"] = dx["DIAGNOSIS"].str.strip()
dx["date_dx"] = pd.to_datetime(
    dx["EXAMDATE"].str.strip(), format="%Y-%m-%d", errors="coerce"
)

dx = dx.loc[
    dx["PTID"].isin(ptids_359)
    & dx["DIAGNOSIS"].isin(["1", "2", "3"])
    & dx["date_dx"].notna(),
    ["PTID", "date_dx", "DIAGNOSIS"],
]

# Ecarter les visites a codes contradictoires (meme regle que tes notebooks).
conflits = dx.groupby(["PTID", "date_dx"])["DIAGNOSIS"].nunique()
conflits = conflits[conflits > 1].index
idx = pd.MultiIndex.from_frame(dx[["PTID", "date_dx"]])
dx = dx.loc[~idx.isin(conflits)].drop_duplicates()

dx["code"] = dx["DIAGNOSIS"].astype(int)
print("Participants avec historique :", dx["PTID"].nunique(), "/ 359")
print("Visites diagnostiques retenues :", len(dx))

In [ ]:
# A.2 Diagnostic de depart (plus ancienne date) et d'arrivee (plus recente).
ordre = dx.sort_values(["PTID", "date_dx"])
premier = ordre.drop_duplicates("PTID", keep="first")[["PTID", "date_dx", "code"]]
dernier = ordre.drop_duplicates("PTID", keep="last")[["PTID", "date_dx", "code"]]

cibles = premier.rename(columns={"date_dx": "date_debut", "code": "dx_debut"}).merge(
    dernier.rename(columns={"date_dx": "date_fin", "code": "dx_fin"}),
    on="PTID", validate="one_to_one",
)

# Pire code atteint sur tout le suivi (si le dernier n'est pas le plus severe).
code_max = dx.groupby("PTID")["code"].max().rename("dx_max")
cibles = cibles.merge(code_max, on="PTID", validate="one_to_one")
cibles["suivi_jours"] = (cibles["date_fin"] - cibles["date_debut"]).dt.days

# --- Etiquettes de classification (tu choisiras laquelle utiliser) ---
# 1) Progression : le diagnostic s'aggrave entre le debut et la fin.
cibles["progression"] = (cibles["dx_fin"] > cibles["dx_debut"]).astype(int)

# 2) Declin a tout moment : le pire code depasse celui du depart (plus sensible).
cibles["declin_a_tout_moment"] = (cibles["dx_max"] > cibles["dx_debut"]).astype(int)

# 3) Conversion MCI -> demence : uniquement pour ceux qui partent MCI (code 2).
cibles["conversion_MCI_demence"] = pd.NA
mci = cibles["dx_debut"].eq(2)
cibles.loc[mci, "conversion_MCI_demence"] = (
    cibles.loc[mci, "dx_max"] >= 3
).astype("Int64")

libelle = {1: "CN", 2: "MCI", 3: "Demence"}
cibles["label_debut"] = cibles["dx_debut"].map(libelle)
cibles["label_fin"] = cibles["dx_fin"].map(libelle)

In [ ]:
# A.3 Ajouter la cible de regression (MMSE a 1 an) pour le sous-groupe de 112.
mmse_cible = manifeste_112[["PTID", "cible_MMSE_1an"]].copy()
mmse_cible["PTID"] = mmse_cible["PTID"].str.strip().str.upper()
mmse_cible["cible_MMSE_1an"] = pd.to_numeric(
    mmse_cible["cible_MMSE_1an"], errors="coerce"
)
cibles = cibles.merge(mmse_cible, on="PTID", how="left", validate="one_to_one")

print("Diagnostic de depart :")
print(cibles["label_debut"].value_counts().to_string())
print("\nProgression (declin debut->fin) :")
print(cibles["progression"].value_counts().to_string())
print("\nDeclin a tout moment :")
print(cibles["declin_a_tout_moment"].value_counts().to_string())
print("\nConversion MCI->demence (parmi les MCI de depart) :")
print(cibles["conversion_MCI_demence"].value_counts(dropna=False).to_string())
print("\nParticipants avec cible MMSE-1an :",
      int(cibles["cible_MMSE_1an"].notna().sum()), "(attendu : 112)")

SORTIE = C.sortie("preparation_modele")
cibles.to_csv(SORTIE / "cibles_par_participant.csv", index=False, encoding="utf-8-sig")
print("\nCibles enregistrees :", SORTIE / "cibles_par_participant.csv")
display(cibles.head())

In [ ]:
# B.1 Date de reference = premiere visite IRM retenue.
visites = pd.read_csv(C.VISITES_359, dtype={"PTID": "string"}, parse_dates=["date_irm"])
visites["PTID"] = visites["PTID"].str.strip().str.upper()
reference = (
    visites.loc[visites["PTID"].isin(ptids_359)]
    .groupby("PTID")["date_irm"].min()
    .dt.normalize().rename("date_reference").reset_index()
)
print("Participants avec date de reference :", len(reference), "/ 359")

In [ ]:
# B.2 Demographie : sexe, niveau d'etudes, age a la reference.
def premier_non_vide(serie):
    s = serie.str.strip().replace("", pd.NA).dropna()
    return s.iloc[0] if len(s) else pd.NA

demo = pd.read_csv(C.table("All_Subjects_PTDEMOG_"), dtype="string", encoding="utf-8-sig")
demo["PTID"] = demo["PTID"].str.strip().str.upper()

demo_agg = demo.groupby("PTID").agg(
    sexe=("PTGENDER", premier_non_vide),
    education=("PTEDUCAT", premier_non_vide),
    naissance_brut=("PTDOBYY", premier_non_vide),
)
tab = reference.merge(demo_agg, on="PTID", how="left")

# Diagnostic : a quoi ressemble la colonne de naissance ?
print("Exemples PTDOBYY :", demo["PTDOBYY"].dropna().unique()[:10].tolist())

# Extraire une annee a 4 chiffres quel que soit le format.
annee = tab["naissance_brut"].astype("string").str.extract(r"(\d{4})")[0]
tab["annee_naissance"] = pd.to_numeric(annee, errors="coerce")
tab["age_reference"] = tab["date_reference"].dt.year - tab["annee_naissance"]
tab["sexe"] = pd.to_numeric(tab["sexe"], errors="coerce")
tab["education"] = pd.to_numeric(tab["education"], errors="coerce")
print("Age calcule pour :", int(tab["age_reference"].notna().sum()), "participants")
print("Age min/median/max :",
      tab["age_reference"].min(), tab["age_reference"].median(), tab["age_reference"].max())

In [ ]:
# B.3 APOE : nombre d'alleles e4 (0, 1 ou 2) — facteur de risque majeur.
def compter_e4(g):
    if pd.isna(g) or "/" not in str(g):
        return pd.NA
    return sum(1 for a in str(g).split("/") if a.strip() == "4")

apoe = pd.read_csv(C.table("All_Subjects_APOERES_"), dtype="string", encoding="utf-8-sig")
apoe["PTID"] = apoe["PTID"].str.strip().str.upper()
apoe_agg = (
    apoe.groupby("PTID")["GENOTYPE"].agg(premier_non_vide)
    .rename("APOE_genotype").reset_index()
)
apoe_agg["APOE_e4"] = apoe_agg["APOE_genotype"].map(compter_e4).astype("Int64")
tab = tab.merge(apoe_agg, on="PTID", how="left")
print("Genotype APOE connu pour :", int(tab["APOE_genotype"].notna().sum()), "participants")

In [ ]:
# B.4 Scores cognitifs a la reference (le plus proche a +/- 180 jours).
def score_a_reference(prefixe, col_date, col_score, nom, tol_jours=180):
    t = pd.read_csv(C.table(prefixe), dtype="string", encoding="utf-8-sig")
    t["PTID"] = t["PTID"].str.strip().str.upper()
    t["date"] = pd.to_datetime(t[col_date].str.strip(), errors="coerce").dt.normalize()
    t[nom] = pd.to_numeric(t[col_score], errors="coerce")
    t = (
        t.loc[t["PTID"].isin(ptids_359)]
        .dropna(subset=["PTID", "date", nom])[["PTID", "date", nom]]
        .drop_duplicates()
    )
    lien = pd.merge_asof(
        reference.sort_values("date_reference"),
        t.sort_values("date"),
        by="PTID", left_on="date_reference", right_on="date",
        direction="nearest", tolerance=pd.Timedelta(days=tol_jours),
    )
    return lien[["PTID", nom]]

scores = [
    ("All_Subjects_MMSE_", "VISDATE", "MMSCORE",  "MMSE_debut"),
    ("All_Subjects_CDR_",  "VISDATE", "CDGLOBAL", "CDR_global"),
    ("All_Subjects_CDR_",  "VISDATE", "CDRSB",    "CDR_SB"),
    ("All_Subjects_ADAS_", "VISDATE", "TOTAL13",  "ADAS13"),
    ("All_Subjects_MOCA_", "VISDATE", "MOCA",     "MOCA_debut"),
    ("All_Subjects_FAQ_",  "VISDATE", "FAQTOTAL", "FAQ_debut"),
]

for prefixe, cd, cs, nom in scores:
    tab = tab.merge(score_a_reference(prefixe, cd, cs, nom), on="PTID", how="left")
    print(f"{nom:12s} : {int(tab[nom].notna().sum())} participants")

In [ ]:
# B.5 Biomarqueurs (optionnels, souvent manquants) : plasma + LCR, +/- 365 jours.
def biomarqueur_a_reference(prefixe, col_date, colonnes, tol_jours=365):
    t = pd.read_csv(C.table(prefixe), dtype="string", encoding="utf-8-sig")
    t["PTID"] = t["PTID"].str.strip().str.upper()
    t["date"] = pd.to_datetime(t[col_date].str.strip(), errors="coerce").dt.normalize()
    for c in colonnes:
        t[c] = pd.to_numeric(t[c], errors="coerce")
    t = (
        t.loc[t["PTID"].isin(ptids_359)]
        .dropna(subset=["PTID", "date"])[["PTID", "date"] + colonnes]
        .drop_duplicates()
    )
    lien = pd.merge_asof(
        reference.sort_values("date_reference"),
        t.sort_values("date"),
        by="PTID", left_on="date_reference", right_on="date",
        direction="nearest", tolerance=pd.Timedelta(days=tol_jours),
    )
    return lien[["PTID"] + colonnes]

plasma = biomarqueur_a_reference(
    "All_Subjects_UPENN_PLASMA_", "EXAMDATE",
    ["pT217_F", "AB42_AB40_F", "NfL_F", "GFAP_F"],
)
tab = tab.merge(plasma, on="PTID", how="left")

lcr = biomarqueur_a_reference(
    "All_Subjects_UPENNBIOMK_ROCHE_", "EXAMDATE",
    ["ABETA42", "TAU", "PTAU"],
)
tab = tab.merge(lcr, on="PTID", how="left")
print("Biomarqueurs plasma/LCR joints.")

In [ ]:
# B.6 Taux de remplissage, puis enregistrement des deux tableaux finaux.
remplissage = pd.DataFrame({
    "variable": tab.columns,
    "valeurs_presentes": [int(tab[c].notna().sum()) for c in tab.columns],
})
remplissage["taux_%"] = (remplissage["valeurs_presentes"] / len(tab) * 100).round(1)
print("Remplissage des variables (sur", len(tab), "participants) :")
display(remplissage.sort_values("taux_%", ascending=False).reset_index(drop=True))

tab.to_csv(SORTIE / "variables_tabulaires.csv", index=False, encoding="utf-8-sig")

# Jeu final : cibles + variables, une ligne par participant.
jeu = cibles.merge(
    tab.drop(columns=["date_reference"]),
    on="PTID", how="left", validate="one_to_one",
)
jeu.to_csv(SORTIE / "jeu_modelisation_359.csv", index=False, encoding="utf-8-sig")
print("\nJeu de modelisation :", jeu.shape)
print("Fichier :", SORTIE / "jeu_modelisation_359.csv")
display(jeu.head())